# Kaggle — entrenamiento (F2.8)

Plantilla derivada de `kaggle_setup.ipynb`. Antes de correr:

1. **Add Data → Your Datasets → `melanoma-isic2020-splits`** (manifiesto, splits, `SHA256SUMS`).
2. **Add Data → Your Datasets → `melanoma-isic2020-512`** (imágenes redimensionadas, F2.0).
3. **Accelerator → GPU**, Internet activado.
4. **Add-ons → Secrets → `WANDB_API_KEY`** (la llave nunca va en el notebook ni en el repositorio).

El código que corre es literalmente el del repositorio en el commit `REPO_SHA`: una sola fuente de
verdad, y la corrida queda atada a ese commit (W&B registra el SHA y el hash de los splits).
Solo se leen `train.txt` y `val.txt`.


In [ ]:
# Parámetros de la corrida
REPO_URL = "https://github.com/Edgar-Ontiveros/melanoma-triage.git"
REPO_SHA = "main"  # ← SHA del commit a ejecutar, nunca `main` en una corrida real
# Overrides de Hydra. Ejemplos:
#   B1 con pos_weight, semilla 0:   ["+experiment=b1_resnet50_224", "train.seed=0"]
#   B1 con muestreo ponderado:      ["+experiment=b1_resnet50_224_sampler", "train.seed=0"]
#   F2.6 condición B:               ["+experiment=prep_b_divide255"]
OVERRIDES = ["+experiment=b1_resnet50_224", "train.seed=0"]
RUN_TAG = "b1-s0"
SPLITS_DIR = "/kaggle/input/datasets/edgaronti26/melanoma-isic2020-splits"
IMAGES_DIR = "/kaggle/input/datasets/edgaronti26/melanoma-isic2020-512"
WORK = "/kaggle/working"

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

subprocess.run(["git", "clone", "--quiet", REPO_URL, f"{WORK}/melanoma"], check=True)
subprocess.run(["git", "-C", f"{WORK}/melanoma", "checkout", "--quiet", REPO_SHA], check=True)
os.chdir(f"{WORK}/melanoma")
SHA = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("commit", SHA)
# Dependencias del grupo `train` (torch ya viene en la imagen de Kaggle;
# timm/lightning/albumentations no siempre).
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", ".[train]"], check=True)

In [ ]:
# Verificar los hashes de los splits contra SHA256SUMS. Falla si no coinciden.
from melanoma.data import verify_split_hashes

hashes = verify_split_hashes(SPLITS_DIR)
print("splits íntegros:", hashes)
assert Path(IMAGES_DIR).exists(), f"no está adjunto el dataset de imágenes: {IMAGES_DIR}"
n_jpg = sum(1 for _ in Path(IMAGES_DIR).glob("*.jpg"))
print(f"{n_jpg} imágenes en {IMAGES_DIR}")

In [ ]:
# Llave de W&B desde Kaggle Secrets. Si no existe, la corrida sigue en modo offline.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B: llave cargada desde Secrets")
except Exception as exc:
    print("W&B sin llave (modo offline):", exc)

In [ ]:
# Entrenar con configs/ del repositorio. Las rutas de Kaggle vienen de data.paths.kaggle
# (data.env=auto detecta el entorno).
run_dir = f"{WORK}/runs/{RUN_TAG}-{SHA}"
cmd = [
    sys.executable,
    "scripts/train.py",
    *OVERRIDES,
    f"data.paths.kaggle.images_dir={IMAGES_DIR}",
    f"data.paths.kaggle.splits_dir={SPLITS_DIR}",
    f"data.paths.kaggle.manifest_path={SPLITS_DIR}/isic2020.csv",
    "data.num_workers=4",
    f"hydra.run.dir={run_dir}",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Artefactos en /kaggle/working: checkpoint, metrics.json, curvas, figuras y resumen.
import json

summary = json.loads(Path(run_dir, "metrics.json").read_text())
print(
    json.dumps(
        {k: summary[k] for k in ("run_name", "best_checkpoint", "epochs_run", "collapse_epochs")},
        indent=2,
    )
)
print(json.dumps(summary["metrics"], indent=2))
print(Path(run_dir, "summary.md").read_text())

## Después de la corrida

Descargar `metrics.json`, `val_predictions.csv`, `summary.md` y `figures/` del directorio de la corrida
(`/kaggle/working/runs/...`) y copiarlos a `reports/runs/<run_name>/` en el repositorio para armar
`reports/f2_baselines.md` y `reports/preprocessing_experiment.md` (`scripts/f2_report.py`).
El checkpoint (`.ckpt`) no se versiona.
